# Predictive probability fo FEV1 data with changing HFEV1 prior


In [1]:
import model_validation.model_evidence as me
import data.breathe_data as bd
import cfr.cfr_helpers as cfrh
import models.helpers as mh
import itertools
import numpy as np
import pandas as pd
import inference.helpers as ih
from plotly.subplots import make_subplots
import viz.viz_helpers as vh
import data.helpers as dh

# Model evidence for publication

For Breathe & CF registry data

Compute log P(FEV1|HFEV1_baseline)

Compute log P(FEV1|HFEV1_predicted)

Expectation: log prob of predictions is higher (less negative)

### CFR data

In [2]:
# CFR data
df_cfr = bd.load_meas_from_excel(
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

In [3]:
# Compute airway resistance

In [4]:
# log prob FEV1 data = log prob of observing every FEV1 values

df_cfr[["log_P_FEV1_HFEV1FT", "P(FEV1|HFEV1FT)"]] = df_cfr.apply(
    lambda row: me.get_log_prob_fev1_data_1_day_model_different_hfev1_priors_for_row(
        row, fev1_col="FEV1", hfev1_prior={"type": "custom", "p": row["P(HFEV1|bFEV1)"]}
    ),
    axis=1,
).apply(pd.Series)

# df_cfr[["log_P_FEV1_HFEV1ST", "P(FEV1|HFEV1ST)"]] = df_cfr.apply(
#     lambda row: me.get_log_prob_fev1_data_1_day_model_different_hfev1_priors_for_row(
#         row, fev1_col="FEV1", hfev1_prior={"type": "custom", "p": row["P(HFEV1|FEV1)"]}
#     ),
#     axis=1,
# ).apply(pd.Series)

df_cfr[["log_P_FEV1_HFEV1", "P(FEV1|HFEV1)"]] = df_cfr.apply(
    lambda row: me.get_log_prob_fev1_data_1_day_model_different_hfev1_priors_for_row(
        row, fev1_col="FEV1"
    ),
    axis=1,
).apply(pd.Series)

In [ ]:
df_cfr["log_P_FEV1_HFEV1FT"].sum()

-8528.245889620002

In [ ]:
# P(FEV1|HFEV1ST) is smaller because the plateau is just where it has to be, not longer than necessary
# (which leads p weight to be distributed amongst more bins, as when observing bFEV1)
df_cfr["log_P_FEV1_HFEV1ST"].sum()

-8502.53927945791

In [ ]:
df_cfr["log_P_FEV1_HFEV1"].sum()

-8766.779296313283

In [ ]:
df_cfr.shape

(2037, 30)

In [ ]:
# df_cfr.to_csv("ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc_with_log_probs.csv", index=False)

#### Study max min diff

In [5]:
df_cfr["log prob FT-NT"] = df_cfr["log_P_FEV1_HFEV1FT"] - df_cfr["log_P_FEV1_HFEV1"]

In [39]:
idx_min_diff = df_cfr["log prob FT-NT"].idxmin()

idx = idx_min_diff
id, height, age, sex = df_cfr.iloc[idx][["ID", "Height", "Age", "Sex"]]

ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
hfev1prior = {"type": "default", "height": height, "age": age, "sex": sex}
HFEV1 = mh.SharedVariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=hfev1prior)

fig = make_subplots(rows=2, cols=2, shared_yaxes=True)
# Prior P(HFEV1|age, sex, height)
ih.plot_histogram(
    fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 1, 1, name=f"prior", colour="lightblue"
)
ih.plot_histogram(
    fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 1, 2, name=f"prior", colour="lightblue"
)

# P(HFEV1FT | bFEV1)
p = df_cfr.iloc[idx]["P(HFEV1|FEV1)"]
ih.plot_histogram(fig, HFEV1, p, 0, HFEV1.b, 1, 1, name=f"P(HFEV1|FEV1)")
fig.data[-1].marker.color = "#0072b2"

# P(FEV1 | HFEV1)
p1 = df_cfr.iloc[idx]["P(FEV1|HFEV1)"]
p2 = df_cfr.iloc[idx]["P(FEV1|HFEV1FT)"]
pmax = max(max(p1), max(p2))
fev1_obs = df_cfr.iloc[idx]["FEV1"]
ih.plot_histogram(
    fig, ecFEV1, p1, ecFEV1.a, ecFEV1.b, 2, 1, name=f"P(FEV1|HFEV1)", colour="#0072b2"
)
vh.add_fev1_prct_pred_line(fig, fev1_obs, pmax, 2, 1, width=1)

# P(HFEV1FT | bFEV1)
p = df_cfr.iloc[idx]["P(HFEV1|bFEV1)"]
ih.plot_histogram(fig, HFEV1, p, 0, HFEV1.b, 1, 2, name=f" P(HFEV1FT|bFEV1)")
fig.data[-1].marker.color = "#d55e00"

# P(FEV1 | HFEV1FT)
ih.plot_histogram(
    fig, ecFEV1, p2, ecFEV1.a, ecFEV1.b, 2, 2, name=f"P(FEV1|HFEV1FT)", colour="#d55e00"
)
vh.add_fev1_prct_pred_line(fig, fev1_obs, pmax, 2, 2, width=1)

title = f"ID {id}, idx {idx}, bFEV1={df_cfr.iloc[idx]['best FEV1']:.2f}, FEV1={df_cfr.iloc[idx]['FEV1']:.2f}<br>log prob diff {df_cfr.iloc[idx]['log prob FT-NT']:.2f}, log prob FEV1|HFEV1 {df_cfr.iloc[idx]['log_P_FEV1_HFEV1']:.2f}, log prob FEV1|HFEV1FT {df_cfr.iloc[idx]['log_P_FEV1_HFEV1FT']:.2f}"
fig.update_layout(title=title, barmode="overlay", height=400, width=1000)
fig.show()

### Breathe data

In [2]:
# Breathe data
df_br = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


In [3]:
# Compute P(HFEV1|bFEV1)

# Get 3rd max row per ID
df_rmax = (
    df_br.sort_values(by=["FEV1", "FEF2575", "O2 Saturation"], ascending=False)
    .groupby("ID")
    .agg(lambda df: df.head(3).tail(1))
    .reset_index()
)

# Calc P(HFEV1|bFEV1) - same for each ID
df_rmax["P(HFEV1|bFEV1)"] = df_rmax.apply(
    lambda row: cfrh.infer_truncated_hfev1_1day_model(row, fev1_col="ecFEV1 (L)"),
    axis=1,
)
df_rmax["P(HFEV1|bFEV1)"].isna().sum()

0

In [ ]:
# log prob FEV1 data given HFEV1FT


# Remove top 3 rows per ID
def get_top_3_fev1_indices_per_ID(df):
    return df.groupby("ID").apply(
        lambda group: group.sort_values(
            by=["FEV1", "FEF2575", "O2 Saturation"],
            ascending=False,
        )
        .head(3)
        .index,
        include_groups=False,
    )


top_3_idx_per_id = list(
    itertools.chain.from_iterable(get_top_3_fev1_indices_per_ID(df_br))
)
assert len(top_3_idx_per_id) <= 3 * df_br["ID"].nunique()
df_br_excl_max = df_br.drop(index=top_3_idx_per_id)


# Compute log prob of FEV1 data for breathe data using HFEV1 prior from dr_rmax_rows
def get_breathe_data_log_prob_HFEV1FT_for_ID(df_excl_max, df_rmax):
    id = df_excl_max.name
    hfev1_prior = df_rmax[df_rmax["ID"] == id]["P(HFEV1|bFEV1)"].values[0]

    return df_excl_max.apply(
        lambda row: pd.Series(
            me.get_log_prob_fev1_data_1_day_model_different_hfev1_priors_for_row(
                row,
                fev1_col="ecFEV1 (L)",
                hfev1_prior={"type": "custom", "p": hfev1_prior},
                debug=False,
            ),
            index=["log P(FEV1|HFEV1FT)", "P(FEV1|HFEV1FT)"],
        ),
        axis=1,
    )


df_br_excl_max.loc[:, ["log P(FEV1|HFEV1FT)", "P(FEV1|HFEV1FT)"]] = (
    df_br_excl_max.groupby("ID").apply(
        lambda df: get_breathe_data_log_prob_HFEV1FT_for_ID(df, df_rmax)
    )
    # .loc assignment uses index alignment, removing the groupby key level and preserving original indexing
    .reset_index(level=0, drop=True)
)

In [33]:
df_br_excl_max[["log P(FEV1|HFEV1)", "P(FEV1|HFEV1)"]] = df_br_excl_max.apply(
    lambda row: me.get_log_prob_fev1_data_1_day_model_different_hfev1_priors_for_row(
        row, fev1_col="ecFEV1 (L)"
    ),
    axis=1,
).apply(pd.Series)

In [ ]:
# df_br_excl_max.to_csv(
#     f"{dh.get_path_to_main()}/ExcelFiles/BR/log_prob_fev1_data_changing_hfev1_priors_1daymodel.csv",
#     index=False,
# )

In [35]:
df_br_excl_max["log P(FEV1|HFEV1)"].sum()

-171773.13037274496

In [37]:
df_br_excl_max["log P(FEV1|HFEV1FT)"].sum()

-166334.58420540206

In [64]:
df_br_excl_max.shape

(40269, 24)

#### Compare log prob FEV1 data through HFEV1 priors with direct FEV1 inference - can't compare becasue not same best FEV1

2-day model with bFEV1 should be equal to 1-day model with HFEV1FT


In [38]:
df_legacy = dh.load_excel(
    f"{dh.get_path_to_main()}/ExcelFiles/BR/log_prob_fev1_data.xlsx",
    date_cols=["Date Recorded"],
)

In [ ]:
# Merge on ID, Date Recorded
df_compare = df_br_excl_max.merge(df_legacy, on=["ID", "Date Recorded"], how="left")
df_compare["diff"] = (
    df_compare["log P(FEV1|HFEV1FT)"] - df_compare["log P(FEV1|2-day FEV1)"]
)

In [ ]:
df_compare["diff"].value_counts()

diff
 0.000000    1584
-0.003541     447
 0.132428     425
-0.165690     289
-0.165577     243
             ... 
 0.055527       1
 0.055527       1
 0.122385       1
 0.122385       1
 0.059574       1
Name: count, Length: 3102, dtype: int64

In [58]:
df_br_excl_max.columns

Index(['ID', 'Date Recorded', 'FEV1', 'O2 Saturation', 'FEF2575', 'ecFEV1',
       'ecFEF2575', 'Sex', 'Height', 'Age', 'Predicted FEV1',
       'Healthy O2 Saturation', 'ecFEV1 % Predicted', 'FEV1 % Predicted',
       'O2 Saturation % Healthy', 'ecFEF2575%ecFEV1', 'idx ecFEV1 (L)',
       'idx O2 saturation (%)', 'idx ecFEF2575%ecFEV1',
       'idx ecFEF25-75 % ecFEV1 (%)', 'log P(FEV1|HFEV1FT)', 'P(FEV1|HFEV1FT)',
       'log P(FEV1|HFEV1)', 'P(FEV1|HFEV1)'],
      dtype='object')

In [ ]:
idx = 0  # df_compare[df_compare['diff'] == 0].index
id = df_compare.loc[idx, "ID"]
print(df_rmax[df_rmax["ID"] == id][["ID", "FEV1"]])
df_compare.loc[
    idx,
    [
        "ID",
        "Date Recorded",
        "FEV1",
        "log P(FEV1|HFEV1FT)",
        "log P(FEV1|2-day FEV1)",
        "diff",
    ],
]

    ID  FEV1
0  101  1.78


ID                               101
Date Recorded             2019-01-25
FEV1                            1.31
log P(FEV1|HFEV1FT)        -4.127317
log P(FEV1|2-day FEV1)     -4.127317
diff                             0.0
Name: 0, dtype: object

## Testing significance

In [8]:
df_br_excl_max = bd.load_meas_from_excel(
    "log_prob_fev1_data_changing_hfev1_priors_1daymodel",
    bypass_sanity_checks=True,
    study_folder="BR",
    use_csv=True,
)

df_br_excl_max["log prob diff"] = (
    df_br_excl_max["log P(FEV1|HFEV1FT)"] - df_br_excl_max["log P(FEV1|HFEV1)"]
)

In [9]:
import numpy as np
import plotly.express as px
import scipy.stats as stats

# 1. Vos données (différences de log-probabilités)
data = df_cfr["log prob FT-NT"].values
data = df_br_excl_max["log prob diff"].values

for data in [df_cfr["log prob FT-NT"].values, df_br_excl_max["log prob diff"].values]:
    print(f"Data: mean={data.mean():.2f}, std={data.std():.2f}, n={len(data)}")
    # 2. Calcul des quantiles théoriques et observés
    osm, _ = stats.probplot(data, dist="norm")
    x_theoretical = osm[0]
    y_observed = osm[1]

    # Trouver les limites globales communes pour avoir des axes identiques
    min_val = min(x_theoretical.min(), y_observed.min())
    max_val = max(x_theoretical.max(), y_observed.max())

    # Margins
    axis_min = np.floor(min_val) - 0.5
    axis_max = np.ceil(max_val) + 0.5

    fig = px.scatter(
        x=x_theoretical, 
        y=y_observed, 
        labels={"x": "Theoretical Quantiles", "y": "Observed Quantiles"}
    )

    # Ref line
    fig.add_shape(
        type="line",
        x0=axis_min, y0=axis_min,
        x1=axis_max, y1=axis_max,
        line=dict(color="red", dash="dash", width=2),
    )

    tick_spacing = 1

    fig.update_layout(
        title="Q-Q Plot",
        template="plotly_white",
        width=600,
        height=600, 
        
        xaxis=dict(
            range=[axis_min, axis_max],
            tickmode='linear',
            tick0=0,
            dtick=tick_spacing,
            gridcolor='rgba(200, 200, 200, 0.3)'
        ),
        
        yaxis=dict(
            range=[axis_min, axis_max],
            scaleanchor="x",
            scaleratio=1,
            tickmode='linear',
            tick0=0,
            dtick=tick_spacing,
            gridcolor='rgba(200, 200, 200, 0.3)'
        )
    )

    fig.show()

# Q-Q measures
# Left from mean there is no tail. 
# Right from mean it's a ehavier than gaussian - bigger win than gaussians

Data: mean=0.12, std=0.35, n=2037


Data: mean=0.14, std=0.45, n=40269


In [11]:
# Paired tests
import numpy as np
import scipy.stats as stats

for data in [df_cfr["log prob FT-NT"].values, df_br_excl_max["log prob diff"].values]:
    # 1. Sign Test (Testing if the median is greater than 0)
    # counts number of positive differences vs negative differences
    positives = np.sum(data > 0)
    negatives = np.sum(data < 0)
    n_total = positives + negatives
    
    binom_res = stats.binomtest(positives, n=n_total, p=0.5, alternative='greater')
    sign_test_p = binom_res.pvalue
    sign_test_stat = binom_res.statistic  # Proportion of positives (k/n)

    # 2. Permutation Test (Testing if the mean difference is significantly > 0)
    def statistic(x):
        return np.mean(x)

    # We use alternative='greater' because you want to test if Model 2 improves over Model 1
    perm_res = stats.permutation_test((data,), statistic, permutation_type='samples', alternative='greater')
    permutation_p = perm_res.pvalue
    permutation_stat = perm_res.statistic  # Observed mean difference

    print(f"Sign Test:")
    print(f"  Count of positives: {positives} out of {n_total}")
    print(f"  Statistic (proportion): {sign_test_stat:.5f}")
    print(f"  p-value: {sign_test_p:.5f}")
    
    print(f"Permutation Test:")
    print(f"  Statistic (mean difference): {permutation_stat:.5f}")
    print(f"  p-value: {permutation_p:.5f}")
    print("-" * 40)

Sign Test:
  Count of positives: 1711 out of 2037
  Statistic (proportion): 0.83996
  p-value: 0.00000
Permutation Test:
  Statistic (mean difference): 0.11710
  p-value: 0.00010
----------------------------------------
Sign Test:
  Count of positives: 35843 out of 40269
  Statistic (proportion): 0.89009
  p-value: 0.00000
Permutation Test:
  Statistic (mean difference): 0.13506
  p-value: 0.00010
----------------------------------------
